# Issue #208 — dynamic N×K and risk-conditioned parlay policy frontier

Contract BJ-UNIFIED-DYNAMIC-N-001 / v3 (REWORK v2). Primary candidate: per-issue learned action-value policy Q_theta(x,a) (MLPRegressor) on the bounded N×K×lane×capital catalog; controls = Ridge + best fixed action; 16 preregistered hash seeds aggregated. SIMPLE_ACTIVE capital = frozen #207 EQUAL_1PCT semantics. 2024/2025/2026 outcomes embargoed. This notebook reads the committed immutable package under `results/issue-208/` — no worker-local files.

In [1]:
from pathlib import Path
import json
import pandas as pd

def _find_root():
    p = Path.cwd()
    for _ in range(6):
        if (p / "results" / "issue-208").exists():
            return p
        p = p.parent
    return Path.cwd()

ROOT = _find_root()
PACKAGE = ROOT / "results" / "issue-208"
import sys
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("package:", PACKAGE.resolve(), "exists:", PACKAGE.exists())

package: /home/drwho1985/workspace/soccer/beijing-lot-bj208-r1/results/issue-208 exists: True


## 1. Data census

171 development issues (CV_2021 58 / CV_2022 58 / CV_2023 55) + 75 LOCKED_TEST (embargoed). Panels built from the frozen #205 value matrix (sha 8fdace9c…); upstream #207 manifest (48c11d97…) bound at load.

In [2]:
census = pd.read_csv(PACKAGE / "census.csv")
census.groupby("split").agg(issues=("issue", "nunique"), mean_M=("M", "mean")).round(1)

,issues,mean_M
split,,
CV_2021,58,321.1
CV_2022,58,285.8
CV_2023,55,283.5
LOCKED_TEST,75,226.2


## 2. Action frontier (counterfactual label surface)

CV_2021 fit fold surface: per-issue × 168 actions, seed-aggregated matched pre-tax capital lift vs frozen N7/K8 incumbent.

In [3]:
surface = pd.read_parquet(PACKAGE / "label_surface_fit.parquet")
print("surface rows:", len(surface))
surface.groupby(["lane", "capital"]).agg(actions=("action_key", "count"), mean_lift=("matched_lift_mean", "mean")).round(1)

surface rows: 9744


actions  mean_lift
lane        capital                          
LEARNED_MLP NATIVE_2_YUAN     1218        9.3
            SIMPLE_ACTIVE     1218     -137.5
MARKET_WOR  NATIVE_2_YUAN     1218       -8.6
            SIMPLE_ACTIVE     1218     -653.8
MARKET_WR   NATIVE_2_YUAN     1218       -9.8
            SIMPLE_ACTIVE     1218     -668.8
VALUE_WOR   NATIVE_2_YUAN     1218      -13.5
            SIMPLE_ACTIVE     1218     -808.9

## 3. N×K performance surface (CV_2021 fit fold)

Median matched lift over lanes/capital, aggregated by N and K.

In [4]:
pivot = surface.pivot_table(index="n", columns="k", values="matched_lift_mean", aggfunc="median").round(1)
pivot

k,8,16,32
n,,,
4,0.0,-16.0,-48.0
5,0.0,-16.0,-48.0
6,0.0,-16.0,-48.0
7,0.0,-16.0,-48.0
8,0.0,-16.0,-48.0
9,-487.0,-495.0,-511.0
10,-487.0,-495.0,-511.0


## 4. Chronological learned policy (safety-gated)

Fold 1 (train CV_2021 → test CV_2022) and Fold 2 (train CV_2021+CV_2022 → test CV_2023). Chronological validation block = last 15% of train issues by issue id; sklearn row-level early stopping disabled (REWORK v2 / D3).

In [5]:
for fold in ("fold1", "fold2"):
    learned = pd.read_parquet(PACKAGE / f"path_learned_{fold}.parquet")
    print(fold, "ending:", round(float(learned.iloc[-1]["post_issue_bankroll_yuan"]), 2), "issues:", len(learned))
loss = pd.read_csv(PACKAGE / "chronological_train_val_loss.csv")
loss

fold1 ending: 99072.0 issues: 58
fold2 ending: 99312.89 issues: 55


,model,fit_issues,validation_issues,fit_rmse,validation_rmse
0,MLP,98,18,1976.327679,1555.211853


## 5. Controls / attribution matrix (CV_2023 test split)

All arms start ¥100,000. capital_change_identical now binds frozen #207 EQUAL_1PCT identities (ends ¥64,659.14 — matches the #207 ledger to 7 decimals; the R1 ¥16-funding bug is fixed, D2).

In [6]:
import glob
rows = []
for f in sorted(glob.glob(str(PACKAGE / "path_control_*_fold2.parquet"))):
    p = pd.read_parquet(f)
    name = Path(f).name.removeprefix("path_control_").removesuffix("_fold2.parquet")
    rows.append({"control": name, "ending_yuan": round(float(p.iloc[-1]["post_issue_bankroll_yuan"]), 2)})
pd.DataFrame(rows).sort_values("ending_yuan", ascending=False).reset_index(drop=True)

,control,ending_yuan
0,static_n8k8,100809.55
1,fixed_n7_dynamic_k,100164.42
2,static_n5k8,100028.00
3,cash_no_bet,100000.00
4,dynamic_nk_fixed_generator,99872.46
5,static_n4k8,99872.46
6,dynamic_n_fixed_k8_lane,99872.46
7,n7k16_sensitivity,99572.50
8,static_n9k8,99556.04
9,static_n6k8,99531.54


## 6. Learning diagnostics

Calibration of predicted vs realized best lift is monotone (Q has signal); action frequency shows the ungated model prefers N4/K32/LEARNED_MLP; OOD/fallback census shows the conservative gate vetoes all 113 test executions → learned path ≡ incumbent.

In [7]:
calib = pd.read_csv(PACKAGE / "calibration_curve.csv")
calib.round(1)

,bin,predicted_best_lift,realized_best_lift,issues
0,0,0.4,2415.4,24
1,1,88.1,4529.2,23
2,2,174.0,8474.0,23
3,3,260.2,11206.6,23
4,4,444.1,27104.6,23


In [8]:
freq = pd.read_csv(PACKAGE / "action_frequency.csv")
freq.head(6)

,action_key,n,k,lane,capital,count,share
0,N4/K32/LEARNED_MLP/NATIVE_2_YUAN,4,32,LEARNED_MLP,NATIVE_2_YUAN,44,0.389381
1,N8/K8/MARKET_WR/NATIVE_2_YUAN,8,8,MARKET_WR,NATIVE_2_YUAN,9,0.079646
2,N7/K8/MARKET_WR/NATIVE_2_YUAN,7,8,MARKET_WR,NATIVE_2_YUAN,9,0.079646
3,N5/K8/MARKET_WR/NATIVE_2_YUAN,5,8,MARKET_WR,NATIVE_2_YUAN,9,0.079646
4,N4/K8/MARKET_WR/NATIVE_2_YUAN,4,8,MARKET_WR,NATIVE_2_YUAN,7,0.061947
5,N9/K8/MARKET_WR/NATIVE_2_YUAN,9,8,MARKET_WR,NATIVE_2_YUAN,6,0.053097


In [9]:
fallback = pd.read_csv(PACKAGE / "ood_fallback_census.csv")
fallback

,split,issues,executed,fallback_incumbent,no_bet
0,CV_2022,58,0,58,58
1,CV_2023,55,0,55,55


## 7. Selection receipt and #209 handoff

Lexicographic rule over 2 constructible folds: learned matched lift = 0.0 on both folds (safety-gated to incumbent), so no row qualifies → freeze N7/K8/MARKET_WOR/native-¥2 incumbent. Handoff carries fitted MLP weights + scaler (label-blind replay).

In [10]:
selection = json.loads((PACKAGE / "selection.json").read_text())
print("eligible:", selection["eligible"])
print("median matched lift:", selection["median_matched_lift_yuan"])
print("positive folds:", selection["positive_fold_count"], "/", selection["constructible_fold_count"])
print("frozen candidate:", selection["frozen_candidate"]["policy"])
handoff = json.loads((PACKAGE / "handoff_209.json").read_text())
print("handoff model_serialized:", handoff["model"].get("model_serialized"))
print("handoff scaler_mean present:", handoff["model"].get("scaler_mean") is not None)
print("handoff model_file:", handoff["model_file"])

eligible: False
median matched lift: 0.0
positive folds: 0 / 2
frozen candidate: FROZEN_N7_INCUMBENT
handoff model_serialized: True
handoff scaler_mean present: True
handoff model_file: results/issue-208/models/fold2_mlp.joblib


## 8. Package verification + tamper self-tests (REQ-E06)

Production verifier: manifest full coverage, receipt binding, provenance identity, capital accounting (start ¥100,000, ¥2 units, continuity), embargo/2024 checks. All four tamper attacks (mutate settlement / self-consistent ledger / forged manifest / forged provenance) must be DETECTED by the production verifier.

In [11]:
from parlay_policy.verify import verify_manifest, verify_capital_accounting, verify_no_embargoed_split, verify_no_2024_issue, verify_receipt, verify_provenance
import json

expected = json.loads((PACKAGE / "SHA256SUMS.json").read_text())
failures = verify_manifest(PACKAGE, expected)
print("manifest failures:", failures[:3] if failures else "NONE")
failures += verify_receipt(PACKAGE)
print("receipt failures:", failures[:3] if failures else "NONE")
ledger = pd.read_parquet(PACKAGE / "issue_capital_ledger.parquet")
failures += verify_capital_accounting(ledger)
failures += verify_no_embargoed_split(ledger)
failures += verify_no_2024_issue(ledger)
print("capital/embargo failures:", failures[:3] if failures else "NONE")
print("TOTAL verification failures:", len(failures))

manifest failures: NONE
receipt failures: NONE
capital/embargo failures: NONE
TOTAL verification failures: 0


In [12]:
tamper = json.loads((PACKAGE / "tamper_results.json").read_text())
for res in tamper:
    print(res["attack"], "->", "DETECTED" if res["detected"] else "MISSED", res["failures"][:1])

mutate_settlement -> DETECTED ['hash mismatch issue_capital_ledger.parquet: 50916c778a2e2d964eacb611c70954b322f5feab6759c2261deb3edbb11380c9 != 9eb15552ad592cc208dfc148d669e8615f3fb5d692b9a3ad4a9dbb9d0a49d81d']
self_consistent_ledger -> DETECTED ['hash mismatch issue_capital_ledger.parquet: 2c1bdb18f3c7fb570952db041724bf019b975e8f61c1bcbc35a3cb4873721ad5 != 9eb15552ad592cc208dfc148d669e8615f3fb5d692b9a3ad4a9dbb9d0a49d81d']
forge_manifest -> DETECTED ['hash mismatch decision_fold1.parquet: d592e56718bb117f5d3bf81ce8c0fb06a59f5c912ac986a19346c7cc7e4e65f8 != 0000000000000000000000000000000000000000000000000000000000000000']
forge_source_trace -> DETECTED ['hash mismatch provenance.json: 926f494d3f583dde66e98d0d04ca52bfb20ddb1ffb8896819400bb7f5c8e1913 != ab51eadf933b6a8bee64ddf98e518564d73595af6d1f49d30e5e3bc06c38363e']


## 9. Boundary statement

- 2024 calendar year: embargoed (no 2024 issue appears in any #208 artifact; real 2024-mutation invariance test passes).
- 2025/2026 LOCKED_TEST: embargoed.
- No production betting or release authority follows from this research.